In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from api.data_utils import extract_from_json, aggregate_to_operator_day, generate_clingo_facts, generate_physical_instance

sns.set_theme(style="whitegrid")


In [2]:
json_files = glob.glob("data/*.json")
df_raw = extract_from_json(json_files)
df_base = aggregate_to_operator_day(df_raw)

In [3]:
def chronological_split_and_clean(df, train_ratio=0.8):
    """
    Ordina per data, splitta temporalmente e rimuove feature con varianza zero.
    """
    # Ordinamento cronologico
    df_sorted = df.sort_values('planning_date').reset_index(drop=True)
    split_idx = int(len(df_sorted) * train_ratio)
    
    df_train = df_sorted.iloc[:split_idx].copy()
    df_test = df_sorted.iloc[split_idx:].copy()
    
    # Rimozione date (non più necessarie dopo lo split)
    df_train = df_train.drop(columns=['planning_date'])
    df_test = df_test.drop(columns=['planning_date'])
    
    # Ricerca feature costanti sul train set
    constant_cols = [col for col in df_train.columns if df_train[col].nunique() <= 1]
    
    if constant_cols:
        df_train = df_train.drop(columns=constant_cols)
        df_test = df_test.drop(columns=constant_cols, errors='ignore')
        print(f"Rimosse feature costanti: {constant_cols}")
        
    print(f"Train set: {len(df_train)} record.")
    print(f"Test set:  {len(df_test)} record.")
    
    return df_train, df_test

# Esecuzione
df_train, df_test = chronological_split_and_clean(df_base)

Train set: 1738 record.
Test set:  435 record.


In [4]:
def process_outliers_and_correlation(df_train, df_test, target='target_assignments', corr_thresh=0.90):
    """
    Rimuove outlier sul target (solo dal Train) e feature altamente correlate (da entrambi).
    """
    # 1. Filtro IQR (Solo Train)
    Q1 = df_train[target].quantile(0.25)
    Q3 = df_train[target].quantile(0.75)
    IQR = Q3 - Q1
    
    mask = df_train[target].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    outliers_count = (~mask).sum()
    df_train_clean = df_train[mask].copy()
    print(f"Outlier rimossi dal target (Train): {outliers_count}")

    # 2. Filtro Multicollinearità (Calcolato sul Train, applicato a entrambi)
    features_for_corr = df_train_clean.drop(columns=['operator_id', target]).select_dtypes(include=[np.number])
    corr_matrix = features_for_corr.corr().abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > corr_thresh)]
    
    if to_drop:
        df_train_clean = df_train_clean.drop(columns=to_drop)
        df_test = df_test.drop(columns=to_drop)
        print(f"Feature scartate per multicollinearità (> {corr_thresh}): {to_drop}")
    else:
        print("Nessuna feature scartata per multicollinearità.")
        
    return df_train_clean, df_test

df_train_corr, df_test_corr = process_outliers_and_correlation(df_train, df_test)

Outlier rimossi dal target (Train): 0
Nessuna feature scartata per multicollinearità.


In [5]:
def final_encoding_and_export(df_train, df_test, cat_thresh=0.05, output_dir="data"):
    """
    Gestisce categorie rare, esegue il One-Hot Encoding e salva i CSV.
    """
    os.makedirs(output_dir, exist_ok=True)
    df_tr = df_train.copy()
    df_te = df_test.copy()
    
    # 1. Raggruppamento categorie rare
    cat_cols = df_tr.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in cat_cols:
        freq = df_tr[col].value_counts(normalize=True)
        frequent_cats = freq[freq >= cat_thresh].index.tolist()
        
        df_tr[col] = df_tr[col].where(df_tr[col].isin(frequent_cats), 'Other')
        df_te[col] = df_te[col].where(df_te[col].isin(frequent_cats), 'Other')

    # 2. One-Hot Encoding Sicuro
    cols_to_encode = [c for c in cat_cols if c != 'operator_id']
    
    df_tr_encoded = pd.get_dummies(df_tr, columns=cols_to_encode, drop_first=True, dtype=int)
    df_te_encoded = pd.get_dummies(df_te, columns=cols_to_encode, drop_first=True, dtype=int)
    
    # 3. Allineamento Colonne (Garantisce che Train e Test abbiano le stesse esatte feature)
    df_tr_encoded, df_te_encoded = df_tr_encoded.align(df_te_encoded, join='left', axis=1, fill_value=0)
    
    # 4. Esportazione
    train_path = f"{output_dir}/train_dataset.csv"
    test_path = f"{output_dir}/test_dataset.csv"
    
    df_tr_encoded.to_csv(train_path, index=False)
    df_te_encoded.to_csv(test_path, index=False)
    
    print(f"Pipeline completata con successo! Dati pronti per il Machine Learning.")
    print(f"- Train CSV salvato in: {train_path} ({df_tr_encoded.shape[1]} feature)")
    print(f"- Test CSV salvato in:  {test_path} ({df_te_encoded.shape[1]} feature)")
    
    return df_tr_encoded, df_te_encoded

df_train_final, df_test_final = final_encoding_and_export(df_train_corr, df_test_corr)

Pipeline completata con successo! Dati pronti per il Machine Learning.
- Train CSV salvato in: data/train_dataset.csv (13 feature)
- Test CSV salvato in:  data/test_dataset.csv (13 feature)


/var/folders/wg/0cp_65ts2slf57w15ldqqn9h0000gn/T/ipykernel_85360/1600382120.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_tr.select_dtypes(include=['object', 'category']).columns.tolist()


In [6]:
from IPython.display import display

def generate_statistical_report(df, title="Report Statistico"):
    """
    Genera un DataFrame riassuntivo con tipi di dato, valori mancanti e distribuzioni.
    """
    print(f"\n--- {title} ---")
    report_data = []
    
    for col in df.columns:
        n_missing = df[col].isnull().sum()
        pct_missing = (n_missing / len(df)) * 100
        n_unique = df[col].nunique()
        dtype = str(df[col].dtype)
        
        # Logica di visualizzazione della distribuzione
        if pd.api.types.is_numeric_dtype(df[col]) and n_unique > 2:
            dist_info = f"Min: {df[col].min():.1f} | Media: {df[col].mean():.1f} | Max: {df[col].max():.1f}"
        else:
            # Per le variabili categoriche o booleane
            top_vals = df[col].value_counts().head(3).to_dict()
            dist_info = "Top: " + ", ".join([f"{k} ({v})" for k, v in top_vals.items()])
            
        report_data.append({
            'Feature': col,
            'Tipo': dtype,
            'Distinti': n_unique,
            'Mancanti': f"{n_missing} ({pct_missing:.1f}%)",
            'Distribuzione': dist_info
        })
        
    report_df = pd.DataFrame(report_data)
    return report_df

# stats_base = generate_statistical_report(df_base, "Situazione Iniziale (Operatore-Giorno)")
# display(stats_base)

# stats_final = generate_statistical_report(df_train_final, "Situazione Finale (Train Set Encoded)")
# display(stats_final)

In [7]:
def plot_features(df, features_to_plot=None, title_suffix="", save_dir="images"):
    """
    Genera grafici per le feature indicate e li salva nella directory specificata.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Se non specifichiamo le feature, le prende tutte tranne l'ID e la data
    cols = features_to_plot if features_to_plot is not None else [c for c in df.columns if c not in ['operator_id', 'planning_date']]
    
    for col in cols:
        if col not in df.columns:
            continue
            
        col_data = df[col].dropna()
        if len(col_data) == 0:
            continue
            
        plt.figure(figsize=(8, 5))
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        is_discrete = df[col].nunique() < 10  # Soglia per trattarla come categorica visiva
        
        plot_title = f"{col} {title_suffix}".strip()
        
        if is_numeric and not is_discrete:
            # Istogramma continuo (es. density_ratio)
            sns.histplot(col_data, kde=True, color="skyblue", bins=20)
            plt.axvline(col_data.mean(), color='red', linestyle='--', label=f'Media: {col_data.mean():.1f}')
            plt.ylabel('Frequenza (Giornate-Operatore)')
            plt.legend()
        else:
            # Grafico a barre
            ax = sns.countplot(data=df, x=col, palette="Set2", order=df[col].value_counts().index)
            plt.xticks(rotation=45)
            plt.ylabel('Conteggio')
            
            # Etichette sopra le barre
            for p in ax.patches:
                height = int(p.get_height())
                if height > 0:
                    ax.annotate(f'{height}', 
                                (p.get_x() + p.get_width() / 2., height), 
                                ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')
                    
        plt.title(plot_title, fontsize=14, pad=15)
        plt.xlabel(col)
        plt.tight_layout()
        
        file_suffix = f"_{title_suffix.strip().replace(' ', '_')}" if title_suffix else ""
        plt.savefig(f"{save_dir}/{col}{file_suffix}.png", bbox_inches='tight')
        plt.show()

# print("GENERAZIONE GRAFICI: PRIMA")
# plot_features(df_base, title_suffix="(Pre-Pulizia)")

# features_chiave_modificate = ['target_assignments', 'density_ratio'] 
# print("GENERAZIONE GRAFICI: DOPO")
# plot_features(df_train_final, features_to_plot=features_chiave_modificate, title_suffix="(Post-Pulizia)")